In [2]:
%pip install pandas numpy joblib scikit-learn

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.0 MB 726.2 kB/s eta 0:00:13
   -- ------------------------------------- 0.5/10.0 MB 726.2 kB/s eta 0:00:13
   -- ------------------------------------- 0.5/10.0 MB 726.2 kB/s eta 0:00:13
   -- ------------------------------------- 0.5/10.0 MB 726.2 kB/s eta 0:00:13
   --- ------------------------------------ 0.8/10.0 MB 434.2 kB/s eta 0:00:22
   --- ------------------------------------ 0.8/10.0 MB 434.2 kB/s eta 0:00:22
   --- ------------------------------------ 0.8/10.0 MB 434.2 kB/s eta 0:00:22
   --- ------------------------------------ 0.8/10.0 MB 434.2 kB/s eta 0:00:22
   ---- ----------------------------------- 1.0/10.0 MB 397.9 kB/s eta 0:00:23
   ---- ----------------------------------- 1.0/10.0 MB 397.9 kB/s eta 0:00:23


In [3]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [4]:
import pandas as pd

people = pd.read_csv("../dataset/archive (9)/01_people.csv")
abilities = pd.read_csv("../dataset/archive (9)/02_abilities.csv")
education = pd.read_csv("../dataset/archive (9)/03_education.csv")
experience = pd.read_csv("../dataset/archive (9)/04_experience.csv")
person_skills = pd.read_csv("../dataset/archive (9)/05_person_skills.csv")
skills = pd.read_csv("../dataset/archive (9)/06_skills.csv")

In [5]:
print(people.shape)
print(abilities.shape)
print(education.shape)
print(experience.shape)
print(person_skills.shape)
print(skills.shape)

(54933, 5)
(1219473, 2)
(75999, 5)
(265404, 6)
(2483376, 2)
(226760, 1)


In [7]:
people.columns
abilities.columns
education.columns
experience.columns
person_skills.columns
skills.columns

Index(['skill'], dtype='str')

In [8]:
people.isnull().sum()

person_id        0
name           114
email        53340
phone        53100
linkedin     46395
dtype: int64

In [9]:
people.drop_duplicates(inplace=True)
abilities.drop_duplicates(inplace=True)
education.drop_duplicates(inplace=True)
experience.drop_duplicates(inplace=True)
person_skills.drop_duplicates(inplace=True)
skills.drop_duplicates(inplace=True)

In [10]:
people.fillna("", inplace=True)
abilities.fillna("", inplace=True)
education.fillna("", inplace=True)
experience.fillna("", inplace=True)
person_skills.fillna("", inplace=True)
skills.fillna("", inplace=True)

,skill
0,Mongo DB-3.2
1,JNDI LDAP
2,Stored Procedures
3,Perform ad-hoc analysis
4,Monitored and resolved flight crew legality is...
...,...
226755,Retention through VE-135 (6 mos after graduati...
226756,Remedy Service Management Tool
226757,Offshore Management
226758,virus and malware removal


In [11]:

people.fillna(0, inplace=True)

,person_id,name,email,phone,linkedin
0,1,Database Administrator,,,
1,2,Database Administrator,,,
2,3,Oracle Database Administrator,,,
3,4,Amazon Redshift Administrator and ETL Develope...,,,
4,5,Scrum Master Scrum Master Scrum Master,,,
...,...,...,...,...,...
54928,54929,Lead Python Developer,,,
54929,54930,Full Stack Python Developer,,,
54930,54931,Eli Lilly,,,
54931,54932,Python Developer,,,


In [12]:
text_columns = people.select_dtypes(include="object").columns

for col in text_columns:
    people[col] = people[col].str.lower()

C:\Users\RAKSHITA\AppData\Local\Temp\ipykernel_18220\4169914680.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = people.select_dtypes(include="object").columns


In [13]:
for col in text_columns:
    people[col] = people[col].str.strip()

In [14]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

for col in text_columns:
    people[col] = people[col].apply(clean_text)

In [15]:
print(people.columns.tolist())

['person_id', 'name', 'email', 'phone', 'linkedin']


In [16]:
print("People")
print(people.columns)

print("\nAbilities")
print(abilities.columns)

print("\nEducation")
print(education.columns)

print("\nExperience")
print(experience.columns)

print("\nPerson Skills")
print(person_skills.columns)

print("\nSkills")
print(skills.columns)

People
Index(['person_id', 'name', 'email', 'phone', 'linkedin'], dtype='str')

Abilities
Index(['person_id', 'ability'], dtype='str')

Education
Index(['person_id', 'institution', 'program', 'start_date', 'location'], dtype='str')

Experience
Index(['person_id', 'title', 'firm', 'start_date', 'end_date', 'location'], dtype='str')

Person Skills
Index(['person_id', 'skill'], dtype='str')

Skills
Index(['skill'], dtype='str')


In [17]:

person_skill_names = person_skills.merge(
    skills,
    on="skill",
    how="left"
)

In [18]:
skills_per_person = person_skill_names.groupby("person_id")["skill"] \
                                      .apply(lambda x: " ".join(x)) \
                                      .reset_index()

In [19]:
people = people.merge(
    skills_per_person,
    on="person_id",
    how="left"
)

In [20]:

print(people.columns.tolist())
print(abilities.columns.tolist())
print(education.columns.tolist())
print(experience.columns.tolist())
print(person_skills.columns.tolist())
print(skills.columns.tolist())

['person_id', 'name', 'email', 'phone', 'linkedin', 'skill']
['person_id', 'ability']
['person_id', 'institution', 'program', 'start_date', 'location']
['person_id', 'title', 'firm', 'start_date', 'end_date', 'location']
['person_id', 'skill']
['skill']


In [21]:
education_group = (
    education.groupby("person_id")["program"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

education_group.head()

,person_id,program
0,1,Bachelor of Science
1,2,bsc in computer science
2,3,Master of Computer Applications in Science and...
3,4,Bachelor in Computer Science
4,5,


In [22]:
people = people.merge(
    education_group,
    on="person_id",
    how="left"
)

people.head()

,person_id,name,email,phone,linkedin,skill,program
0,1,database administrator,,,,Database administration Database Ms sql server...,Bachelor of Science
1,2,database administrator,,,,sql server management studio visual studio sql...,bsc in computer science
2,3,oracle database administrator,,,,DATABASES ORACLE (4 years) ORACLE 10G SQL LINU...,Master of Computer Applications in Science and...
3,4,amazon redshift administrator and etl develope...,,,,Maintain multiple database environments (Redsh...,Bachelor in Computer Science
4,5,scrum master scrum master scrum master,,,,Scrum Agile software development Product backl...,


In [23]:
experience_group = (
    experience.groupby("person_id")
    .agg({
        "title": lambda x: " ".join(x.astype(str)),
        "firm": lambda x: " ".join(x.astype(str))
    })
    .reset_index()
)

experience_group.head()

,person_id,title,firm
0,1,Database Administrator Database Administrator,Family Private Care LLC Incomm
1,2,Database Administrator,Intercontinental Registry
2,3,Oracle Database Administrator Oracle Database ...,Cognizant Convergys
3,4,Amazon Redshift Administrator and ETL Develope...,"MSP Recovery - Fort Lauderdale, FL CEAACES - Q..."
4,5,Scrum Master Oracle Database Administrator/ Sc...,Quest Technologies Prudential Time Warner Cable


In [24]:
people = people.merge(
    experience_group,
    on="person_id",
    how="left"
)

people.head()

,person_id,name,email,phone,linkedin,skill,program,title,firm
0,1,database administrator,,,,Database administration Database Ms sql server...,Bachelor of Science,Database Administrator Database Administrator,Family Private Care LLC Incomm
1,2,database administrator,,,,sql server management studio visual studio sql...,bsc in computer science,Database Administrator,Intercontinental Registry
2,3,oracle database administrator,,,,DATABASES ORACLE (4 years) ORACLE 10G SQL LINU...,Master of Computer Applications in Science and...,Oracle Database Administrator Oracle Database ...,Cognizant Convergys
3,4,amazon redshift administrator and etl develope...,,,,Maintain multiple database environments (Redsh...,Bachelor in Computer Science,Amazon Redshift Administrator and ETL Develope...,"MSP Recovery - Fort Lauderdale, FL CEAACES - Q..."
4,5,scrum master scrum master scrum master,,,,Scrum Agile software development Product backl...,,Scrum Master Oracle Database Administrator/ Sc...,Quest Technologies Prudential Time Warner Cable


In [25]:
people.fillna("", inplace=True)

,person_id,name,email,phone,linkedin,skill,program,title,firm
0,1,database administrator,,,,Database administration Database Ms sql server...,Bachelor of Science,Database Administrator Database Administrator,Family Private Care LLC Incomm
1,2,database administrator,,,,sql server management studio visual studio sql...,bsc in computer science,Database Administrator,Intercontinental Registry
2,3,oracle database administrator,,,,DATABASES ORACLE (4 years) ORACLE 10G SQL LINU...,Master of Computer Applications in Science and...,Oracle Database Administrator Oracle Database ...,Cognizant Convergys
3,4,amazon redshift administrator and etl develope...,,,,Maintain multiple database environments (Redsh...,Bachelor in Computer Science,Amazon Redshift Administrator and ETL Develope...,"MSP Recovery - Fort Lauderdale, FL CEAACES - Q..."
4,5,scrum master scrum master scrum master,,,,Scrum Agile software development Product backl...,,Scrum Master Oracle Database Administrator/ Sc...,Quest Technologies Prudential Time Warner Cable
...,...,...,...,...,...,...,...,...,...
54928,54929,lead python developer,,,,Django Angular JS JavaScript JQuery Node.js Py...,,Lead Python Developer Lead Python Developer Sr...,People's United Bank Entrust Guide One Insuran...
54929,54930,full stack python developer,,,,Python Django AWS AngularJS Bootstrap JavaScri...,,Full Stack Python Developer Sr. Python Develop...,"Fresenius Medical Care, MA Hughes Networks Man..."
54930,54931,eli lilly,,,,Python 2.7 HTML5 CSS3 AJAX JSON JQuery Active ...,,Sr. Python Developer Sr. Python Developer Pyth...,Eli Lilly Comcast Core logic Concentrix Insura...
54931,54932,python developer,,,,Python 3.1x PyQuery PyQt Django Angular.js Ope...,,Python Developer Python Developer Python Devel...,Intuit GE Energy Safeway Inc Intralinks Holdin...


In [26]:
people["resume_text"] = (
    people["skill"] + " " +
    people["program"] + " " +
    people["title"] + " " +
    people["firm"]
)

In [27]:
abilities_group = (
    abilities.groupby("person_id")["ability"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

people = people.merge(
    abilities_group,
    on="person_id",
    how="left"
)

people.fillna("", inplace=True)

people["resume_text"] = (
    people["skill"] + " " +
    people["ability"] + " " +
    people["program"] + " " +
    people["title"] + " " +
    people["firm"]
)

In [28]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

people["resume_text"] = people["resume_text"].apply(clean_text)

In [29]:
people.to_csv("final_resume_dataset.csv", index=False)

In [30]:
print(people["title"])

0            Database Administrator Database Administrator
1                                   Database Administrator
2        Oracle Database Administrator Oracle Database ...
3        Amazon Redshift Administrator and ETL Develope...
4        Scrum Master Oracle Database Administrator/ Sc...
                               ...                        
54928    Lead Python Developer Lead Python Developer Sr...
54929    Full Stack Python Developer Sr. Python Develop...
54930    Sr. Python Developer Sr. Python Developer Pyth...
54931    Python Developer Python Developer Python Devel...
54932    MetroBikes Python/Flask Developer Python Devel...
Name: title, Length: 54933, dtype: str
